In [54]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")
ussmallcap = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CUSS ETF Stock Price History.csv")
silver = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SSLN ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope, ussmallcap, silver]

N = len(dfs)
T = len(dfs[0]) - 1

dfs = [df[["Date", "Price", "Vol."]] for df in dfs]

# Convert Date to datetime
dfs_new = []
for df in dfs:
    df["Date"] = pd.to_datetime(df["Date"])
    dfs_new.append(df)

dfs = dfs_new

names = ["snp", "china", "em", "gold", "india", "mscieurope",
         "smallcapeurope", "ussmallcap", "silver"]

dfs_renamed = []

for name, df in zip(names, dfs):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    
    df = df.rename(columns={
        "Price": f"Price_{name}",
        "Vol.": f"Vol_{name}"
    })
    
    dfs_renamed.append(df)

dfs = dfs_renamed

# Merge all on Date
df_merged = dfs[0]
for df in dfs[1:]:
    df_merged = df_merged.merge(df, on="Date", how="inner", suffixes=("", "_x"))

# Sort by date
df_merged = df_merged.sort_values("Date")

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

df_merged = df_merged.set_index("Date")

price_cols = [col for col in df_merged.columns if col.startswith("Price_")]
vol_cols = [col for col in df_merged.columns if col.startswith("Vol_")]

prices_df = df_merged[price_cols].apply(pd.to_numeric, errors="coerce")
volumes_df = df_merged[vol_cols].copy()

for col in volumes_df.columns:
    volumes_df[col] = volumes_df[col].apply(correct_volume)

volumes_df = volumes_df.fillna(0.0)

prices = prices_df.values  # shape: (T+1, N)
volumes = volumes_df.values  # shape: (T+1, N)

returns = prices[1:] / prices[:-1] - 1
prices = prices[1:]
volumes = volumes[1:]         # (T, N)
dates = df_merged.index[1:]       # (T,)
print(returns.shape, prices.shape, volumes.shape, dates.shape)

(2413, 9) (2413, 9) (2413, 9) (2413,)


In [55]:
from rollingwindow_and_eval import rolling_window
from helpers import stress_values_to_dict, costs_to_dict, cash_allocs_to_dict

alpha = 0.05   # tail probability for CVaR
max_cash_alloc = 0.3
min_weight = 0.1
stepsize = 50
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030, 0.050, 0.08]) / 100
risk_free = 0.03
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
window_size = 252
validation_size = 252

lambdas = [0.1, 0.3, 0.5, 0.8, 1.1]
gammas = [0.01, 0.025, 0.04, 0.055, 0.07]
stress_corr_weights = [0.1, 0.2, 0.3]
cash_alloc_params = [0.2, 0.5, 0.8]
m_params = [
    np.array([0.33, 0.33, 0.33]),
    np.array([0.50, 0.25, 0.25]),
    np.array([0.25, 0.50, 0.25]),
    np.array([0.25, 0.25, 0.50]),
]

"""
track:
var_alpha, cvar, sharpe, sortino, mean_return, md
costs, stress values, cash allocs
"""

total_stats_momentum = np.full((9, len(gammas), len(stress_corr_weights), len(m_params)), np.nan) # gamma, stress_corr, mparams
total_stats_momentum_cvar = np.full((9, len(lambdas), len(gammas), len(m_params)), np.nan) # lambda, gamma, mparams
total_stats_er = np.full((9, len(gammas), len(stress_corr_weights), len(cash_alloc_params)), np.nan) # gamma, stress_corr, cash_alloc_param
total_stats_er_cvar = np.full((9, len(lambdas), len(gammas), len(stress_corr_weights)), np.nan) # lambda, gamma, stress_corr
total_stats_sharpe = np.full((9, len(gammas), len(stress_corr_weights), len(cash_alloc_params)), np.nan) # gamma, stress_corr, cash_alloc_param
total_stats_sharpe_cvar = np.full((9, len(lambdas), len(gammas), len(stress_corr_weights)), np.nan) # lambda, gamma, stress_corr
labels = ["ER", "ER_cvar", "sharpe", "sharpe_cvar", "momentum_based", "momentum_cvar",
    "risk_parity", "hrp_weights", "equal_weights"]

for l_idx, l in enumerate(lambdas):
    for g_idx, g in enumerate(gammas):
        for mp_idx, mp in enumerate(m_params):
            for cash_idx, cash in enumerate(cash_alloc_params):
                for stress_corr_idx, stress_corr in enumerate(stress_corr_weights):

                    df, total_costs, total_stress_values, track_pct_rebalances, track_cash_allocs, allweights = rolling_window(
                        returns, volumes, prices,
                        window_size, stepsize, validation_size,
                        g, alpha, l,
                        mp, risk_free, etf_spreads, weight_diff_rebalance,
                        stress_corr, max_cash_alloc, cash, min_weight
                        )
                    
                    mean_weights = np.mean(allweights, axis=0) # shape (num_strategies, num_assets)

                    print('DEBUG EXPECTED RETURN STRATEGY STATISTICS:')
                    print(df.loc["ER", :].values.tolist())
                    print()

                    for stratlabel in labels:
                        assert len(df.loc[stratlabel, :].values.tolist()) == 6

                    values = df.loc["ER", :].values.tolist()
                    print("Assigning:", values[3])  # sortino

                    # weights_momentum[:, g_idx, stress_corr_idx, mp_idx] = mean_weights[0, :]
                    # weights_momentum_cvar[:, g_idx, stress_corr_idx, mp_idx] = mean_weights[1, :]
                    # weights_er[:, g_idx, stress_corr_idx, cash_idx] = mean_weights[2, :]
                    # weights_er_cvar[:, l_idx, g_idx, stress_corr_idx] = mean_weights[3, :]
                    # weights_sharpe[:, g_idx, stress_corr_idx, cash_idx] = mean_weights[4, :]
                    # weights_sharpe_cvar[:, l_idx, g_idx, stress_corr_idx] = mean_weights[5, :]
                    
                    stresses = stress_values_to_dict(labels, total_stress_values)
                    costs = costs_to_dict(labels, total_costs)
                    cash_allocs = cash_allocs_to_dict(labels, track_cash_allocs)
                    
                    total_stats_momentum[:, g_idx, stress_corr_idx, mp_idx] = (
                        df.loc["momentum_based", :].values.tolist()
                        + [stresses["momentum_based"], costs["momentum_based"], cash_allocs["momentum_based"]]
                    )

                    total_stats_momentum_cvar[:, l_idx, g_idx, mp_idx] = (
                        df.loc["momentum_cvar", :].values.tolist()
                        + [stresses["momentum_cvar"], costs["momentum_cvar"], cash_allocs["momentum_cvar"]]
                    )

                    total_stats_er[:, g_idx, stress_corr_idx, cash_idx] = (
                        df.loc["ER", :].values.tolist()
                        + [stresses["ER"], costs["ER"], cash_allocs["ER"]]
                    )

                    total_stats_er_cvar[:, l_idx, g_idx, stress_corr_idx] = (
                        df.loc["ER_cvar", :].values.tolist()
                        + [stresses["ER_cvar"], costs["ER_cvar"], cash_allocs["ER_cvar"]]
                    )

                    total_stats_sharpe[:, g_idx, stress_corr_idx, cash_idx] = (
                        df.loc["sharpe", :].values.tolist()
                        + [stresses["sharpe"], costs["sharpe"], cash_allocs["sharpe"]]
                    )

                    total_stats_sharpe_cvar[:, l_idx, g_idx, stress_corr_idx] = (
                        df.loc["sharpe_cvar", :].values.tolist()
                        + [stresses["sharpe_cvar"], costs["sharpe_cvar"], cash_allocs["sharpe_cvar"]]
                    )



c:\Projects\portfolio\further_opts\objectives.py:27: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



DEBUG EXPECTED RETURN STRATEGY STATISTICS:
[-0.008408617362090838, -0.01425889026898907, 0.5821021498979545, 0.7598131330392661, 0.0003381276566707009, -0.1292239457097282]

Assigning: 0.7598131330392661


c:\Projects\portfolio\further_opts\objectives.py:27: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



DEBUG EXPECTED RETURN STRATEGY STATISTICS:
[-0.008408273413499196, -0.014256481789088636, 0.5821163838117037, 0.7603832893151001, 0.000337526605943767, -0.12920720942973837]

Assigning: 0.7603832893151001


c:\Projects\portfolio\further_opts\objectives.py:27: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



...

DEBUG EXPECTED RETURN STRATEGY STATISTICS:
[-0.007082409223004854, -0.012008244714320824, 0.5788865192518956, 0.7507282262616243, 0.0003037512280668064, -0.12696613816486566]

Assigning: 0.7507282262616243


c:\Projects\portfolio\further_opts\objectives.py:27: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



DEBUG EXPECTED RETURN STRATEGY STATISTICS:
[-0.007090494324426664, -0.012021522600863003, 0.5788865192518956, 0.7507301722371792, 0.00030416404189820804, -0.12696613816486566]

Assigning: 0.7507301722371792


c:\Projects\portfolio\further_opts\objectives.py:27: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



DEBUG EXPECTED RETURN STRATEGY STATISTICS:
[-0.007098492637742652, -0.01203466143361364, 0.5788865192518956, 0.7507914180821301, 0.0003045740904331002, -0.12696613816486566]

Assigning: 0.7507914180821301


In [56]:
from plotly.subplots import make_subplots

stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_momentum

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 3], zmin=zmin, zmax=zmax, showscale=False), row=2, col=2)
    img.update_layout(title=f"{stat} for MOMENTUM - X=gammas, Y=stressCorr")
    img.show()


In [57]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_momentum_cvar

print('mparams used:')
for mp_val in m_params:
    print(mp_val)
for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 3], zmin=zmin, zmax=zmax, showscale=False), row=2, col=2)
    img.update_layout(title=f"{stat} for MOMENTUM WITH CVAR - X=lambdas, Y=gammas")
    img.show()


mparams used:
[0.33 0.33 0.33]
[0.5  0.25 0.25]
[0.25 0.5  0.25]
[0.25 0.25 0.5 ]


In [58]:

stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_er

print("Any NaNs in STRATEGY?", np.isnan(STRATEGY).any())
print("Min raw:", np.min(STRATEGY))

print('cash alloc params used:')
for cp in cash_alloc_params:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for EXPECTED RETURNS - X=gammas, Y=stressCorrs")
    img.show()


Any NaNs in STRATEGY? False
Min raw: -0.1292239457097282
cash alloc params used:
0.2
0.5
0.8


In [59]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_er_cvar

print('stress_corrparams used:')
for cp in stress_corr_weights:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for EXPECTED RETURNS WITH CVAR - X=lambdas, Y=gammas")
    img.show()


stress_corrparams used:
0.1
0.2
0.3


In [62]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_sharpe

print('cash params used:')
for cp in cash_alloc_params:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for SHARPE OPTIMALIZATION - X=gamma, Y=stress")
    img.show()


cash params used:
0.2
0.5
0.8


In [63]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_sharpe_cvar

print('stress values used:')
for cp in stress_corr_weights:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for SHARPE OPTIMALIZATION WITH CVAR - X=lambda, Y=gamma")
    img.show()


stress values used:
0.1
0.2
0.3


In [ ]:
"""
CONCLUSION

FOR MOMENTUM BASED:


FOR SHARPE BASED:



"""